Testing macro feature engine class before deploying...

In [1]:
import os
import sys
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Dynamically locate data warehouse ROOT directory
notebook_path = Path(os.getcwd())
root_dir = notebook_path
while root_dir.name != "data_warehouse" and root_dir.parent != root_dir:
    root_dir = root_dir.parent

# 2. Inject the ROOT path into Python's system path for imports
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# 3. Import your updated structural engine
from utils.macro_feature_engine import MacroFeatureEngine

print(f"Sandbox Environment online. Root Path: {root_dir}")


Sandbox Environment online. Root Path: /Users/bonwier/PythonProjects/data_warehouse


In [2]:
# Connect directly to your newly created, spatial-coordinate-rich transitory project database
project_db_path = root_dir / "databases" / "transitory" / "peri_urban_ag_analysis.db"
project_conn = sqlite3.connect(project_db_path)

# FIXED: Removed 'approvalfy' from the select statement
test_loans_df = pd.read_sql_query(
    "SELECT standardized_fips, naics_4d, is_long_duration, risk_cohort_id "
    "FROM source_loans_snapshot WHERE split_assignment = 'TRAIN';", 
    project_conn
)
project_conn.close()

print(f"Loaded {len(test_loans_df):,} training records carrying standardized FIPS coordinates.")
test_loans_df.head()


Loaded 44,831 training records carrying standardized FIPS coordinates.


,standardized_fips,naics_4d,is_long_duration,risk_cohort_id
0,31111,4452,0,2
1,29095,4244,0,2
2,12111,4452,1,1
3,28049,3118,1,1
4,28049,3118,1,1


In [3]:
# 1. Initialize the engine
engine = MacroFeatureEngine(database_dir=root_dir / "databases")

# 2. Run the enrichment layer letting the engine dynamically pick your max available database year
enriched_test_df = engine.enrich_snapshot_portfolio(
    loan_df=test_loans_df,
    fips_col='standardized_fips',
    irs_vintage_year=None  # FIXED: Triggers the dynamic max-year database check
)

# 3. Clean up resources
engine.close()


print("\nEnrichment complete! Available columns inside sandbox:")
print(enriched_test_df.columns.tolist())


🚀 Enriching portfolio snapshot using the definitive 2022 structural macro cross-section...
 • Enrichment Complete. Attached 4 metrics to 44,831 records.

Enrichment complete! Available columns inside sandbox:
['standardized_fips', 'naics_4d', 'is_long_duration', 'risk_cohort_id', 'macro_wealth_cushion', 'filer_density_velocity', 'filer_density_acceleration', 'msa_wealth_cushion']


In [4]:
# 1. Row Preservation Check
row_delta = len(enriched_test_df) - len(test_loans_df)
print("==================================================================")
print(" METROPOLITAN GRAVITY WELL AUDIT")
print("==================================================================")
print(f" • Row Preservation Status: {'PASSED' if row_delta == 0 else 'FAILED'}")
print(f" • Total Missing (NaN) Values in County Cushions: {enriched_test_df['macro_wealth_cushion'].isna().sum()}")

# Verify if our regional MSA feature populated cleanly
if 'msa_wealth_cushion' in enriched_test_df.columns:
    print(f" • Total Missing (NaN) Values in Regional MSA Cushions: {enriched_test_df['msa_wealth_cushion'].isna().sum()}")
else:
    print(" ⚠️ 'msa_wealth_cushion' column not found in output frame.")

print("\n2. STATISTICAL DISTRIBUTION SNAPSHOT (Structural Features):")
print("------------------------------------------------------------------")
features_to_describe = ['macro_wealth_cushion', 'filer_density_velocity']
if 'msa_wealth_cushion' in enriched_test_df.columns:
    features_to_describe.append('msa_wealth_cushion')

print(enriched_test_df[features_to_describe].describe())

print("\n3. SAMPLE STRUCTURAL CROSS-SECTION INSPECTION:")
print("------------------------------------------------------------------")
enriched_test_df[['standardized_fips', 'naics_4d', 'risk_cohort_id'] + features_to_describe].dropna().head(10)


 METROPOLITAN GRAVITY WELL AUDIT
 • Row Preservation Status: PASSED
 • Total Missing (NaN) Values in County Cushions: 0
 • Total Missing (NaN) Values in Regional MSA Cushions: 0

2. STATISTICAL DISTRIBUTION SNAPSHOT (Structural Features):
------------------------------------------------------------------
       macro_wealth_cushion  filer_density_velocity  msa_wealth_cushion
count          44831.000000            44831.000000        44831.000000
mean               0.049545                0.039805            0.052361
std                0.036415                0.051570            0.017095
min                0.000919               -0.336493            0.026032
25%                0.028936                0.003759            0.042983
50%                0.043124                0.032626            0.047932
75%                0.059478                0.060142            0.054229
max                1.018700                0.391132            0.157621

3. SAMPLE STRUCTURAL CROSS-SECTION INSPECTION

,standardized_fips,naics_4d,risk_cohort_id,macro_wealth_cushion,filer_density_velocity,msa_wealth_cushion
0,31111,4452,2,0.028803,-0.019254,0.044632
1,29095,4244,2,0.036314,0.010969,0.055317
2,12111,4452,1,0.056423,0.218144,0.108655
3,28049,3118,1,0.052994,-0.067181,0.028413
4,28049,3118,1,0.052994,-0.067181,0.028413
5,39049,4452,1,0.041260,0.021611,0.042354
6,40017,3111,1,0.019147,0.212776,0.041325
7,40027,4452,1,0.033369,0.058824,0.041325
8,26163,3118,1,0.051846,0.015556,0.049145
9,39089,3111,1,0.026489,0.037916,0.042354


In [ ]:
# Quick Sandbox Warehouse Audit
spatial_conn = sqlite3.connect(root_dir / "databases" / "spatial_crosswalk.db")
irs_conn = sqlite3.connect(root_dir / "databases" / "irs_county_soi.db")

print("1. Checking available years in your CENSUS_MSA crosswalk table:")
msa_years = pd.read_sql_query("SELECT DISTINCT data_year FROM map_source_to_fips WHERE source_agency='CENSUS_MSA';", spatial_conn)
print(msa_years.to_string(index=False))

print("\n2. Checking sample FIPS values directly from your IRS table:")
irs_fips_sample = pd.read_sql_query("SELECT county_fips, COUNT(*) as record_count FROM county_economics WHERE calendar_year=2021 LIMIT 5;", irs_conn)
print(irs_fips_sample)

spatial_conn.close()
irs_conn.close()


In [ ]:
# Connect directly to the IRS database path based on your architecture
# The Definitive Raw IRS Data Audit
irs_conn = sqlite3.connect(root_dir / "databases" / "irs_county_soi.db")

# 1. Print out a raw 3-row sample of your economics table to see the exact column text names
columns_check = pd.read_sql_query("SELECT * FROM county_economics LIMIT 3;", irs_conn)
print("--- EXACT DATABASE COLUMN NAMES ---")
print(columns_check.columns.tolist())

# 2. Check a raw data slice for a single county across both target years (2022 and 2017)
print("\n--- RAW DATA INSPECTION FOR AUTAUGA COUNTY, AL (FIPS 01001) ---")
raw_slice = pd.read_sql_query(
    "SELECT calendar_year, county_fips, total_returns, wages_and_salaries "
    "FROM county_economics WHERE county_fips IN ('01001', '1001') AND calendar_year IN (2017, 2022);",
    irs_conn,
)
print(raw_slice)

irs_conn.close()